In [1]:
# Cell 1: Environment and GPU check

import sys
import platform
from pathlib import Path

print("Python executable:", sys.executable)
print("Python version:", sys.version)
print("Platform:", platform.platform())

try:
    import torch
    
    print("\nPyTorch version:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    
    if torch.cuda.is_available():
        print("CUDA version used by PyTorch:", torch.version.cuda)
        print("GPU name:", torch.cuda.get_device_name(0))
        print("GPU memory allocated GB:", round(torch.cuda.memory_allocated(0) / 1024**3, 3))
        print("GPU memory reserved GB:", round(torch.cuda.memory_reserved(0) / 1024**3, 3))
    else:
        print("CUDA is not available in this notebook environment.")
        
except ImportError as error:
    print("PyTorch is not installed or not available in this environment.")
    print(error)

Python executable: d:\Projects\evidence_fashion_recommender\.venv\Scripts\python.exe
Python version: 3.11.15 (main, Jun  2 2026, 22:29:49) [MSC v.1944 64 bit (AMD64)]
Platform: Windows-10-10.0.26200-SP0

PyTorch version: 2.11.0+cu128
CUDA available: True
CUDA version used by PyTorch: 12.8
GPU name: NVIDIA GeForce RTX 5070 Ti
GPU memory allocated GB: 0.0
GPU memory reserved GB: 0.0


In [2]:
# Cell 2: Project folders and notebook paths

from pathlib import Path

PROJECT_ROOT = Path.cwd()

DATA_DIR = PROJECT_ROOT / "data"
RAW_DATA_DIR = DATA_DIR / "raw"
PROCESSED_DATA_DIR = DATA_DIR / "processed"
KB_DIR = DATA_DIR / "kb"

OUTPUTS_DIR = PROJECT_ROOT / "outputs"
EMBEDDINGS_DIR = OUTPUTS_DIR / "embeddings"
INDEXES_DIR = OUTPUTS_DIR / "indexes"
RESULTS_DIR = OUTPUTS_DIR / "results"
FIGURES_DIR = OUTPUTS_DIR / "figures"

folders = [
    DATA_DIR,
    RAW_DATA_DIR,
    PROCESSED_DATA_DIR,
    KB_DIR,
    OUTPUTS_DIR,
    EMBEDDINGS_DIR,
    INDEXES_DIR,
    RESULTS_DIR,
    FIGURES_DIR,
]

for folder in folders:
    folder.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("\nCreated/verified folders:")
for folder in folders:
    print("-", folder.relative_to(PROJECT_ROOT))

Project root: d:\Projects\evidence_fashion_recommender

Created/verified folders:
- data
- data\raw
- data\processed
- data\kb
- outputs
- outputs\embeddings
- outputs\indexes
- outputs\results
- outputs\figures


In [3]:
# Cell 3: Core library imports

import os
import json
import random
import math
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image
from tqdm.auto import tqdm

from datasets import load_dataset

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

print("Core imports loaded successfully.")
print("Ready for Stage 0: real dataset loading and inspection.")

Core imports loaded successfully.
Ready for Stage 0: real dataset loading and inspection.


In [4]:
# Cell 4: Load real Polyvore-style dataset from Hugging Face

DATASET_NAME = "Marqo/polyvore"

print(f"Loading dataset: {DATASET_NAME}")

dataset = load_dataset(DATASET_NAME)

print("\nDataset loaded successfully.")
print("Available splits:")
print(dataset)

print("\nSplit names:")
print(list(dataset.keys()))

Loading dataset: Marqo/polyvore



Dataset loaded successfully.
Available splits:
DatasetDict({
    data: Dataset({
        features: ['image', 'category', 'text', 'item_ID'],
        num_rows: 94096
    })
})

Split names:
['data']


In [5]:
# Cell 5: Inspect dataset structure and raw examples

split_names = list(dataset.keys())
main_split_name = split_names[0]

print("Using first split for initial inspection:", main_split_name)

main_split = dataset[main_split_name]

print("\nNumber of rows in selected split:", len(main_split))

print("\nColumn names:")
print(main_split.column_names)

print("\nFeature types:")
print(main_split.features)

print("\nFirst raw example:")
first_example = main_split[0]

for key, value in first_example.items():
    value_type = type(value)
    
    if isinstance(value, str):
        preview = value[:300]
    elif isinstance(value, (list, tuple)):
        preview = value[:3]
    else:
        preview = value
        
    print(f"\n{key}")
    print("Type:", value_type)
    print("Preview:", preview)

print("\nFirst 3 rows as a small pandas preview:")
display(main_split.select(range(min(3, len(main_split)))).to_pandas())

Using first split for initial inspection: data

Number of rows in selected split: 94096

Column names:
['image', 'category', 'text', 'item_ID']

Feature types:
{'image': Image(mode=None, decode=True), 'category': Value('string'), 'text': Value('string'), 'item_ID': Value('string')}

First raw example:

image
Type: <class 'PIL.JpegImagePlugin.JpegImageFile'>
Preview: <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=274x400 at 0x231D92B7CD0>

category
Type: <class 'str'>
Preview: Day Dresses

text
Type: <class 'str'>
Preview: tibi knit long sleeve dress

item_ID
Type: <class 'str'>
Preview: 100002074_1

First 3 rows as a small pandas preview:


,image,category,text,item_ID
0,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x01\x00H\x00H\x00\x00\xff\xdb\x00C\x00\x03\x02\x02\x03\x02\x02\...,Day Dresses,tibi knit long sleeve dress,100002074_1
1,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x01\x00H\x00H\x00\x00\xff\xdb\x00C\x00\x03\x02\x02\x03\x02\x02\...,Boots,michael kors leather over-the-knee boots,100002074_2
2,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x01\x00H\x00H\x00\x00\xff\xdb\x00C\x00\x03\x02\x02\x03\x02\x02\...,Handbags,givenchy leather medium antigona duffel black,100002074_3


In [6]:
# Cell 6: Convert selected split to DataFrame and inspect basic statistics

items_df_raw = main_split.to_pandas()

print("Raw DataFrame shape:", items_df_raw.shape)

print("\nColumns:")
print(list(items_df_raw.columns))

print("\nData types:")
display(items_df_raw.dtypes.to_frame("dtype"))

print("\nMissing values per column:")
missing_summary = (
    items_df_raw
    .isna()
    .sum()
    .sort_values(ascending=False)
    .to_frame("missing_count")
)

missing_summary["missing_percent"] = (
    missing_summary["missing_count"] / len(items_df_raw) * 100
).round(2)

display(missing_summary)

print("\nPossible ID columns:")
possible_id_columns = [
    column for column in items_df_raw.columns
    if "id" in column.lower()
]
print(possible_id_columns)

print("\nPossible category columns:")
possible_category_columns = [
    column for column in items_df_raw.columns
    if any(keyword in column.lower() for keyword in ["category", "cat", "type"])
]
print(possible_category_columns)

print("\nPossible text/metadata columns:")
possible_text_columns = [
    column for column in items_df_raw.columns
    if any(keyword in column.lower() for keyword in ["title", "text", "description", "name", "metadata"])
]
print(possible_text_columns)

print("\nPreview:")
display(items_df_raw.head())

Raw DataFrame shape: (94096, 4)

Columns:
['image', 'category', 'text', 'item_ID']

Data types:


,dtype
image,object
category,str
text,str
item_ID,str



Missing values per column:


,missing_count,missing_percent
image,0,0.0
category,0,0.0
text,0,0.0
item_ID,0,0.0



Possible ID columns:
['item_ID']

Possible category columns:
['category']

Possible text/metadata columns:
['text']

Preview:


,image,category,text,item_ID
0,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x01\x00H\x00H\x00\x00\xff\xdb\x00C\x00\x03\x02\x02\x03\x02\x02\...,Day Dresses,tibi knit long sleeve dress,100002074_1
1,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x01\x00H\x00H\x00\x00\xff\xdb\x00C\x00\x03\x02\x02\x03\x02\x02\...,Boots,michael kors leather over-the-knee boots,100002074_2
2,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x01\x00H\x00H\x00\x00\xff\xdb\x00C\x00\x03\x02\x02\x03\x02\x02\...,Handbags,givenchy leather medium antigona duffel black,100002074_3
3,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x01\x00H\x00H\x00\x00\xff\xdb\x00C\x00\x03\x02\x02\x03\x02\x02\...,Sunglasses,bottega veneta acetate leather sunglasses,100002074_4
4,"{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x01\x01,\x01,\x00\x00\xff\xdb\x00C\x00\x03\x02\x02\x03\x02\x02\...",Floral Decor,pier imports stem,100002074_5


In [7]:
# Cell 7: Build clean item table with outfit IDs

items_df = items_df_raw.copy()

items_df["item_ID"] = items_df["item_ID"].astype(str)
items_df["category"] = items_df["category"].astype(str)
items_df["text"] = items_df["text"].astype(str)

# Marqo/polyvore item_ID format appears to be: outfitID_itemPosition
# Example: 100002074_1 -> outfit_ID = 100002074, item_position = 1
id_parts = items_df["item_ID"].str.rsplit("_", n=1, expand=True)

items_df["outfit_ID"] = id_parts[0]
items_df["item_position"] = pd.to_numeric(id_parts[1], errors="coerce")

# Clean text field for later embedding
items_df["item_text"] = (
    items_df["category"].fillna("") 
    + " | " 
    + items_df["text"].fillna("")
)

# Reorder useful metadata columns first
metadata_columns = ["item_ID", "outfit_ID", "item_position", "category", "text", "item_text", "image"]
items_df = items_df[metadata_columns]

print("Clean item table shape:", items_df.shape)
print("\nPreview:")
display(items_df.head(10))

print("\nNumber of item_ID duplicates:", items_df["item_ID"].duplicated().sum())
print("Number of rows with missing outfit_ID:", items_df["outfit_ID"].isna().sum())
print("Number of rows with missing item_position:", items_df["item_position"].isna().sum())

Clean item table shape: (94096, 7)

Preview:


,item_ID,outfit_ID,item_position,category,text,item_text,image
0,100002074_1,100002074,1,Day Dresses,tibi knit long sleeve dress,Day Dresses | tibi knit long sleeve dress,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x01\x00H\x00H\x00\x00\xff\xdb\x00C\x00\x03\x02\x02\x03\x02\x02\...
1,100002074_2,100002074,2,Boots,michael kors leather over-the-knee boots,Boots | michael kors leather over-the-knee boots,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x01\x00H\x00H\x00\x00\xff\xdb\x00C\x00\x03\x02\x02\x03\x02\x02\...
2,100002074_3,100002074,3,Handbags,givenchy leather medium antigona duffel black,Handbags | givenchy leather medium antigona duffel black,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x01\x00H\x00H\x00\x00\xff\xdb\x00C\x00\x03\x02\x02\x03\x02\x02\...
3,100002074_4,100002074,4,Sunglasses,bottega veneta acetate leather sunglasses,Sunglasses | bottega veneta acetate leather sunglasses,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x01\x00H\x00H\x00\x00\xff\xdb\x00C\x00\x03\x02\x02\x03\x02\x02\...
4,100002074_5,100002074,5,Floral Decor,pier imports stem,Floral Decor | pier imports stem,"{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x01\x01,\x01,\x00\x00\xff\xdb\x00C\x00\x03\x02\x02\x03\x02\x02\..."
5,100002074_7,100002074,7,Coats,miranda coat,Coats | miranda coat,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x01\x00H\x00H\x00\x00\xff\xdb\x00C\x00\x03\x02\x02\x03\x02\x02\...
6,100010727_1,100010727,1,Blazers,three pocket blazer,Blazers | three pocket blazer,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x01\x00`\x00`\x00\x00\xff\xdb\x00C\x00\x03\x02\x02\x03\x02\x02\...
7,100010727_2,100010727,2,Skinny Jeans,givenchy skinny jean,Skinny Jeans | givenchy skinny jean,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x01\x00`\x00`\x00\x00\xff\xdb\x00C\x00\x03\x02\x02\x03\x02\x02\...
8,100010727_4,100010727,4,Watches,guess black silver-tone chronograph watch,Watches | guess black silver-tone chronograph watch,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x01\x00H\x00H\x00\x00\xff\xdb\x00C\x00\x03\x02\x02\x03\x02\x02\...
9,100010727_5,100010727,5,Sunglasses,ray-ban original wayfarer sunglasses,Sunglasses | ray-ban original wayfarer sunglasses,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x01\x00H\x00H\x00\x00\xff\xdb\x00C\x00\x03\x02\x02\x03\x02\x02\...



Number of item_ID duplicates: 0
Number of rows with missing outfit_ID: 0
Number of rows with missing item_position: 0
